In [2]:
!pip install ultralytics
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Criando a pasta destino no Colab
!mkdir -p /content/dataset

# Extraindo o arquivo Projeto_EPI.zip
# Nota: O comando abaixo assume que o arquivo está na raiz do seu Google Drive
!unzip -q "/content/drive/MyDrive/Projeto_EPI.zip" -d /content/dataset

# Verificar o que foi extraído
!ls /content/dataset

unzip:  cannot find or open /content/drive/MyDrive/Projeto_EPI.zip, /content/drive/MyDrive/Projeto_EPI.zip.zip or /content/drive/MyDrive/Projeto_EPI.zip.ZIP.


In [4]:
import yaml
import os

yaml_path = '/content/dataset/data.yaml'

if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # Ajustando os caminhos para o padrão que o Colab entende
    data['train'] = '/content/dataset/train/images'

    # Verifica se a pasta se chama 'valid' ou 'val' e ajusta
    if os.path.exists('/content/dataset/valid'):
        data['val'] = '/content/dataset/valid/images'
    elif os.path.exists('/content/dataset/val'):
        data['val'] = '/content/dataset/val/images'
    else:
        # Se realmente não tiver valid, usamos a de treino para não travar o erro
        data['val'] = '/content/dataset/train/images'

    with open(yaml_path, 'w') as f:
        yaml.dump(data, f)
    print("Configuração do YAML finalizada com sucesso!")
else:
    print("Arquivo data.yaml não encontrado. Verifique se o unzip funcionou.")

Arquivo data.yaml não encontrado. Verifique se o unzip funcionou.


In [5]:
!ls -R /content/

/content/:
dataset  drive

/content/dataset:

/content/drive:
MyDrive  Othercomputers

/content/drive/MyDrive:
'Análise Olímpica USA (1).gsheet'
'Análise Olímpica USA.gsheet'
'Análise Olímpica USA (Resumida).gsheet'
'Bio (1).csv'
'Bio (2).csv'
'Bio (3).csv'
'Bio (4).csv'
'Bio (5).csv'
 Bio.csv
'Bio (USA).csv'
 cadastro_pdvs.parquet
 cadastro_produtos.parquet
'Certificado César-ADA (Ciência de Dados).pdf'
'Certificado de Escolaridade.pdf'
 Classroom
'Colab Notebooks'
'Comprovante de Escolaridade.pdf'
'Comprovante de Residência.pdf'
'Contagem de medalhas dos Jogos Olímpicos.csv'
'CONTRATO DE PRESTAÇÃO DE SERVIÇOS EDUCACIONAIS EM CURSO DE GRADUAÇÃO.pdf'
'CPF (1).pdf'
 CPF.pdf
 CTPSDigital_02746345463_05-12-2024.pdf
'Culto de Santa Ceia Rústico Marrom Cartaz.png'
'Currículo Lattes.pdf'
 Currículo.pdf
'Currículo Profissional.pdf'
'Desafio Prático'
'Desafio Prático.docx'
 desafio_prático.py
 Diploma.pdf
'Documento de Daniel Tavares'
'Documento de Daniel Tavares (1)'
'Do

In [6]:
# Criando a pasta destino
!mkdir -p /content/dataset

# Extraindo o arquivo real que está no seu Drive
!unzip -q "/content/drive/MyDrive/Projeto_EPI/Workers.v1i.yolov8.zip" -d /content/dataset

# Conferindo se agora o data.yaml aparece
!ls /content/dataset

replace /content/dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
data.yaml  README.dataset.txt  README.roboflow.txt  train  valid


In [7]:
from ultralytics import YOLO

# Carregando o modelo inicial
model = YOLO('yolov8n.pt')

# Treinando
model.train(
    data='/content/dataset/data.yaml',
    epochs=50,
    imgsz=640,
    device=0,
    project='Residencia_EPI',
    name='modelo_v1'
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [8]:
import yaml

with open('/content/dataset/data.yaml', 'r') as f:
    config = yaml.safe_load(f)
    print("Conteúdo do YAML atual:")
    print(config)

print("\nVerificando pastas no sistema:")
!ls -d /content/dataset/train/images 2>/dev/null || echo "ERRO: Pasta train/images não encontrada!"
!ls -d /content/dataset/valid/images 2>/dev/null || echo "ERRO: Pasta valid/images não encontrada!"

Conteúdo do YAML atual:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 4, 'names': ['clothes', 'head', 'work_clothes', 'work_hat'], 'roboflow': {'workspace': '001-0i9x0', 'project': 'workers-yy6yy', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/001-0i9x0/workers-yy6yy/dataset/1'}}

Verificando pastas no sistema:
/content/dataset/train/images
/content/dataset/valid/images


In [9]:
import yaml

# Caminho do seu arquivo
yaml_path = '/content/dataset/data.yaml'

# Definindo a configuração correta para o Colab
dados_corrigidos = {
    'path': '/content/dataset', # Caminho base
    'train': 'train/images',    # Caminho relativo ao 'path' acima
    'val': 'valid/images',      # Caminho relativo ao 'path' acima
    'nc': 4,
    'names': ['clothes', 'head', 'work_clothes', 'work_hat']
}

# Gravando o arquivo corrigido
with open(yaml_path, 'w') as f:
    yaml.dump(dados_corrigidos, f, default_flow_style=False)

print("✅ YAML Corrigido!")
# Mostra o conteúdo para conferir
!cat /content/dataset/data.yaml

✅ YAML Corrigido!
names:
- clothes
- head
- work_clothes
- work_hat
nc: 4
path: /content/dataset
train: train/images
val: valid/images


In [10]:
from ultralytics import YOLO

# 1. Carregamos o modelo base (o "estudante")
model = YOLO('yolov8n.pt')

# 2. Iniciamos o treinamento real
model.train(
    data='/content/dataset/data.yaml',
    epochs=50,                # 50 rodadas de aprendizado
    imgsz=640,                # Tamanho da imagem
    device=0,                 # Usa a GPU do Google
    batch=16,                 # Processa 16 fotos por vez
    project='Projeto_EPI',    # Nome da pasta final
    name='experimento_01'     # Versão do seu modelo
)

Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: 0
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


In [11]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Criando a pasta destino
!mkdir -p /content/dataset

# Extraindo o arquivo real que está no seu Drive
!unzip -q "/content/drive/MyDrive/Projeto_EPI/Workers.v1i.yolov8.zip" -d /content/dataset

# Verificando se as pastas train e valid apareceram
!ls /content/dataset

replace /content/dataset/data.yaml? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
data.yaml  README.dataset.txt  README.roboflow.txt  train  valid


In [4]:
import yaml
from ultralytics import YOLO

# 1. Ajuste final do YAML para não ter erro de caminho
yaml_path = '/content/dataset/data.yaml'
config = {
    'path': '/content/dataset',
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 4,
    'names': ['clothes', 'head', 'work_clothes', 'work_hat']
}

with open(yaml_path, 'w') as f:
    yaml.dump(config, f)

print("✅ Tudo pronto! Iniciando o treinamento na GPU...")

# 2. Chamada do modelo
model = YOLO('yolov8n.pt')

# 3. Treino
model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    device=0,   # Agora vai funcionar porque a GPU está ativa!
    batch=16,
    project='Projeto_EPI',
    name='treino_final'
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Tudo pronto! Iniciando o treinamento na GPU...
Ultralytics 8.4.48 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7cb038493290>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [5]:
import shutil

# Copia a pasta de resultados inteira para o seu Drive
shutil.copytree('/content/runs/detect/Projeto_EPI/treino_final', '/content/drive/MyDrive/Projeto_EPI/RESULTADOS_FINAIS')

print("✅ Todos os pesos e gráficos foram salvos no seu Google Drive!")

✅ Todos os pesos e gráficos foram salvos no seu Google Drive!
